## CNN Weights Upload to S3 (GroupFuture)

A compact end-to-end example that builds a small CNN in PyTorch and stores all model weights in AWS S3 via LAILA.

### What this notebook does
- Build a simple CNN model in `torch`
- Collect all weight tensors from `model.state_dict()`
- Wrap each tensor as a LAILA entry (`laila.constant`) to get global IDs
- Upload all entries together in one `laila.memorize(...)` call
- Receive a `GroupFuture`, wait for completion, and inspect status
- Fetch one weight back to verify round-trip integrity

### Credentials
This notebook reads AWS credentials from:
`laila.args`


In [ ]:
%load_ext autoreload
%autoreload 2
 

In [2]:
import sys
sys.path.append("/home/ubuntu/")


In [3]:
import laila
laila.read_args("/home/ubuntu/laila/vault/dev_secrets.toml")
import torch
import torch.nn as nn


/home/ubuntu/laila/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from laila.pool import S3Pool

required = ["AWS_BUCKET_NAME", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION"]
missing = [name for name in required if getattr(laila.args, name, None) is None]
if missing:
    raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
)


In [5]:
laila.add_pool(s3_pool, pool_nickname="cnn_weights_s3_pool")


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((8, 8)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)





In [7]:
#I DO SOME TRAINING HERE
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN

In [8]:
final_model = SimpleCNN().eval()

In [9]:
# Wrap every weight tensor as a separate LAILA constant
from typing import Any


weight_entries = {
    name: laila.constant(data=tensor.detach().cpu(), nickname=name)
    for name, tensor in final_model.state_dict().items()
}

print(f"Prepared {len(weight_entries)} weight entries.")
for name, entry in list[tuple[str, Any]](weight_entries.items()):
    print(name, "->", entry.global_id)


Prepared 8 weight entries.
features.0.weight -> LAILA:ENTRY:GLOBAL_ID:bef5ab85-fba2-50b4-9e33-c9c1a02bbacf
features.0.bias -> LAILA:ENTRY:GLOBAL_ID:363b2cf2-b668-54f5-bb3a-e8b6489b0252
features.2.weight -> LAILA:ENTRY:GLOBAL_ID:0f4ac0d9-cf54-5ec4-a06b-bedcb80fa099
features.2.bias -> LAILA:ENTRY:GLOBAL_ID:8cd0486b-7406-5706-a008-2f805bd9bd69
classifier.1.weight -> LAILA:ENTRY:GLOBAL_ID:fc1bd6cd-42c1-5460-ae37-c92c2ff17549
classifier.1.bias -> LAILA:ENTRY:GLOBAL_ID:55929ea1-9f0d-5a4b-b125-c3a39dfc74db
classifier.3.weight -> LAILA:ENTRY:GLOBAL_ID:71ff4699-4ace-56d1-9f5b-e7775f19e4f6
classifier.3.bias -> LAILA:ENTRY:GLOBAL_ID:6ed197f9-f412-524c-9c2b-ad84640bd0ff


In [10]:
my_model_manifest = laila.constant(
    data = {name:entry.global_id for name, entry in weight_entries.items()},
    nickname = "my_model",
)

In [11]:
# Upload all weights together to get a GroupFuture
weight_group_future = laila.memorize(
    list(weight_entries.values()),
    pool_nickname="cnn_weights_s3_pool",
)

print("Future type:", type(weight_group_future).__name__)
print("Status before wait:", weight_group_future.status)
weight_group_future.wait()
print("Status after wait:", weight_group_future.status)
print("Underlying futures:", len(weight_group_future))


Future type: GroupFuture
Status before wait: {'total': 8.0, 'percentages': {'finished': 0.0, 'running': 0.0, 'not_started': 100.0, 'error': 0.0, 'cancelled': 0.0}}
Status after wait: {'total': 8.0, 'percentages': {'finished': 100.0, 'running': 0.0, 'not_started': 0.0, 'error': 0.0, 'cancelled': 0.0}}
Underlying futures: 8


In [12]:
manifest_future = laila.memorize(
    my_model_manifest,
    pool_nickname="cnn_weights_s3_pool",
)
manifest_future.wait()


In [13]:
# Round-trip check on one weight tensor
sample_name = next(iter(weight_entries.keys()))
sample_entry = weight_entries[sample_name]
original_tensor = final_model.state_dict()[sample_name]

remember_future = laila.remember(sample_entry.global_id, pool_nickname="cnn_weights_s3_pool")
remember_future.wait()
recovered_entry = remember_future.result

assert torch.equal(recovered_entry.data, original_tensor)
print("Recovered tensor matches:", sample_name)


Recovered tensor matches: features.0.weight


In [14]:
# Optional cleanup: delete uploaded weights from S3
forget_group_future = laila.forget(
    [entry.global_id for entry in weight_entries.values()],
    pool_nickname="cnn_weights_s3_pool",
)
forget_group_future.wait()
print("Cleanup complete.")


Cleanup complete.
